In [1]:
!nvidia-smi

Wed Sep 23 03:14:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ortools
!pip install pycuda

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 14.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 w

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.4 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5280314 sha256=47919403176a16d3c37251116cb496a4a44bfb12432ef5c0f54a969c86888e5e
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


In [3]:
from ortools.algorithms.python import knapsack_solver
import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
from pycuda.compiler import SourceModule
import time
import threading

In [4]:
# Colab starts with an empty filesystem, so pull the project in to get problems.py.
# Re-run this cell after a runtime restart: the clone survives, sys.path does not.
import os
import sys

REPO_URL = "https://github.com/andrewrowell/subset-sum-gpu.git"
REPO_DIR = "/content/subset-sum-gpu"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

Cloning into '/content/subset-sum-gpu'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 102 (delta 37), reused 79 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 219.50 KiB | 9.15 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [5]:
NUMBER_OF_TRIALS = 100
PREVIEW_ITEMS = 15

import numpy as np

import problems

# Every implementation in this project solves the identical set of problems by
# reading them from problems.py.
problem_set = problems.generate()

num_problems = problem_set.num_problems
num_items_per_problem = problem_set.num_items
max_capacity = problem_set.max_capacity
capacities = problem_set.capacities

items = problem_set.items

num_items = np.full(num_problems, num_items_per_problem, dtype=np.int32)
max_values = np.zeros(num_problems, dtype=np.int32)

print(f"{num_problems} problems, {num_items_per_problem} items each, capacity at most {max_capacity}")
for i in range(3):
    row = problem_set.items[i]
    head = ", ".join(str(item) for item in row[:PREVIEW_ITEMS])
    rest = f", ... ({len(row) - PREVIEW_ITEMS} more)" if len(row) > PREVIEW_ITEMS else ""
    print(f"Problem {i + 1}: capacity {problem_set.capacities[i]}, items [{head}{rest}]")

10000 problems, 100 items each, capacity at most 100
Problem 1: capacity 82, items [5, 38, 33, 22, 22, 43, 5, 35, 10, 5, 26, 48, 37, 38, 36, ... (85 more)]
Problem 2: capacity 37, items [41, 10, 40, 1, 40, 39, 39, 33, 24, 35, 14, 39, 28, 23, 25, ... (85 more)]
Problem 3: capacity 75, items [18, 45, 25, 35, 23, 14, 38, 48, 13, 39, 13, 36, 39, 23, 37, ... (85 more)]


In [6]:
# OR-Tools Section

# This section should run in Colab T4
import platform
print(platform.node())

# Function to solve a single subset sum problem using OR-Tools
def solve_subset_sum(items, capacity, problem_idx, results):
    solver = knapsack_solver.KnapsackSolver(
        knapsack_solver.KNAPSACK_MULTIDIMENSION_BRANCH_AND_BOUND_SOLVER, 'SubsetSumExample')

    # Subset sum is knapsack with an item's value equal to its weight, so
    # OR-Tools is given the same array for both.
    solver.init(items, [items], [capacity])

    max_value = solver.solve()
    results[problem_idx] = max_value
    #print(f"Problem {problem_idx + 1}: Maximum value = {max_value}")

# Storage for results
results = [0] * num_problems

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Create and start threads
    threads = []
    for i in range(num_problems):
        t = threading.Thread(target=solve_subset_sum, args=(items[i], capacities[i], i, results))
        threads.append(t)
        t.start()

    # Wait for all threads to complete
    for t in threads:
        t.join()

    # Print execution time
    end_time = time.time()
    #print(f"Threaded execution time: {end_time - start_time:.6f} seconds")
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Print the final results
#for i in range(num_problems):
for i in range(10):
    print(f"Final Result for Problem {i + 1}: Maximum value = {results[i]}")

50eac566d8bd
Average CPU execution time: 2.805418 seconds
Final Result for Problem 1: Maximum value = 82
Final Result for Problem 2: Maximum value = 37
Final Result for Problem 3: Maximum value = 75
Final Result for Problem 4: Maximum value = 6
Final Result for Problem 5: Maximum value = 65
Final Result for Problem 6: Maximum value = 98
Final Result for Problem 7: Maximum value = 38
Final Result for Problem 8: Maximum value = 58
Final Result for Problem 9: Maximum value = 74
Final Result for Problem 10: Maximum value = 78


In [7]:
# CUDA kernel to solve multiple subset sum problems. One block solves one
# problem; its threads split the capacities 0..max_capacity between them.
#
# reachable[w] is 1 when some subset of the items seen so far sums to exactly w.
# Adding an item makes w reachable if w - item already was. Each item reads one
# table and writes the other, so no thread can see a value that already includes
# the current item, which would let that item be used twice.
kernel_code = """
__global__ void subset_sum(int *items, int *capacities, int *num_items, int *max_values, int max_capacity) {
    int problem_idx = blockIdx.x;
    int width = max_capacity + 1;

    extern __shared__ unsigned char tables[];
    unsigned char *current = tables;
    unsigned char *next = tables + width;

    // Only the empty subset exists before any item is considered
    for (int w = threadIdx.x; w < width; w += blockDim.x) {
        current[w] = (w == 0);
    }
    __syncthreads();

    int count = num_items[problem_idx];
    int *problem_items = items + problem_idx * count;
    for (int i = 0; i < count; i++) {
        int item = problem_items[i];
        for (int w = threadIdx.x; w < width; w += blockDim.x) {
            next[w] = current[w] | (w >= item ? current[w - item] : 0);
        }
        __syncthreads();

        unsigned char *swap = current;
        current = next;
        next = swap;
    }

    // The answer is the largest reachable sum that fits in the capacity
    if (threadIdx.x == 0) {
        int w = capacities[problem_idx];
        while (!current[w]) {
            w--;
        }
        max_values[problem_idx] = w;
    }
}
"""

# Compile the kernel code
mod = SourceModule(kernel_code)

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Host data is already constructed
    max_values = np.zeros(num_problems, dtype=np.int32)

    # Measure execution time
    start_time = time.time()

    # Allocate memory on the GPU
    items_gpu = cuda.mem_alloc(items.nbytes)
    capacities_gpu = cuda.mem_alloc(capacities.nbytes)
    num_items_gpu = cuda.mem_alloc(num_items.nbytes)
    max_values_gpu = cuda.mem_alloc(max_values.nbytes)

    # Copy data to the GPU
    cuda.memcpy_htod(items_gpu, items)
    cuda.memcpy_htod(capacities_gpu, capacities)
    cuda.memcpy_htod(num_items_gpu, num_items)
    cuda.memcpy_htod(max_values_gpu, max_values)

    # Launch the kernel
    subset_sum = mod.get_function("subset_sum")
    shared_memory_size = 2 * (max_capacity + 1)  # two one-byte tables for one problem
    threads_per_block = min(max_capacity + 1, 1024)  # one thread per capacity, up to the CUDA limit
    subset_sum(items_gpu, capacities_gpu, num_items_gpu, max_values_gpu, np.int32(max_capacity),
            block=(threads_per_block, 1, 1), grid=(num_problems, 1), shared=shared_memory_size)

    # Copy the result back to the CPU
    cuda.memcpy_dtoh(max_values, max_values_gpu)

    # Print execution time
    end_time = time.time()
    #print(f"CUDA execution time: {end_time - start_time:.6f} seconds")
    execution_times.append(end_time - start_time)

print(f"Average PyCUDA execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Check against the OR-Tools section above
mismatches = np.flatnonzero(max_values != np.array(results))
print(f"{len(mismatches)} of {num_problems} results differ from OR-Tools")

# Print the results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: Maximum value = {max_values[i]}")

Average PyCUDA execution time: 0.003016 seconds
0 of 10000 results differ from OR-Tools
Problem 1: Maximum value = 82
Problem 2: Maximum value = 37
Problem 3: Maximum value = 75
Problem 4: Maximum value = 6
Problem 5: Maximum value = 65
Problem 6: Maximum value = 98
Problem 7: Maximum value = 38
Problem 8: Maximum value = 58
Problem 9: Maximum value = 74
Problem 10: Maximum value = 78
